In [49]:
from pathlib import Path
import geopandas as gpd

CONFIG_PATH = Path("configs/ATLAS")
DATA_PATH = CONFIG_PATH / "data"
RUN_PATH = Path("run/")

BUILDINGS_DIR = next(CONFIG_PATH.glob("**/buildings/buildings.shp")) # https://data.cityofchicago.org/Community-Economic-Development/Boundaries-Zoning-Districts-current-/dj47-wfun/about_data
ZONING_DIR = next(DATA_PATH.glob("**/Boundaries*/*.shp"))            # https://data.cityofchicago.org/Buildings/Building-Footprints/syp8-uezg/about_data
RESULTS_DIR = next(RUN_PATH.glob("output.jsonl"))

In [17]:
zoning_gdf = gpd.read_file(ZONGING_DIR)
buildings_gdf = gpd.read_file(BUILDINGS_DIR)

buildings_gdf = buildings_gdf.to_crs(zoning_gdf.crs)
buildings_gdf["geometry"] = buildings_gdf.geometry.centroid

/Users/isaacsalvador/Git/ATLAS/venv/lib/python3.14/site-packages/geopandas/io/file.py:576: UserWarning: Error parsing datetimes, original strings are returned: Out of bounds nanosecond timestamp: 0218-04-17, at position 323. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)
/var/folders/hb/v7xdn2hn4fx_rgmhyl711r900000gn/T/ipykernel_65875/2181225997.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  buildings_gdf["geometry"] = buildings_gdf.geometry.centroid


In [40]:
prefix_map = {
    "RS": "HOME", "RT": "HOME", "RM": "HOME",
    "B1": "DISCRETIONARY", "B2": "DISCRETIONARY", "B3": "DISCRETIONARY",
    "C1": "WORK", "C2": "WORK", "C3": "WORK",
    "M1": "WORK", "M2": "WORK", "M3": "WORK",
}

zoning_gdf["activity_type"] = (
    zoning_gdf["zone_class"]
    .str.extract(r"^([A-Z]+\d?)", expand=False)  # pull prefix
    .map(prefix_map)
    .fillna("DISCRETIONARY")  # PD/PMD/etc default
)

In [43]:
joined = gpd.sjoin(
    buildings_gdf,
    zoning_gdf[["geometry", "activity_type"]],
    how="left",
    predicate="within"
)

In [ ]:
import json

def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

agent_results = read_jsonl(RESULTS_DIR)

In [64]:
import re

def parse_time_sec(time_str, fallback_sec):
    match = re.match(r"(\d{1,2}):(\d{2})\s*(AM|PM)", time_str.strip(), re.IGNORECASE)
    if not match:
        return fallback_sec + 3600
    h, m, period = int(match.group(1)), int(match.group(2)), match.group(3).upper()
    if period == "PM" and h != 12:
        h += 12
    elif period == "AM" and h == 12:
        h = 0
    return h * 3600 + m * 60


def sample_trips(itinerary, buildings_gdf, agent_idx=None):
    locations = itinerary["locations"]
    departure_times = itinerary["departure_times"]

    sampled = {
        act: buildings_gdf[buildings_gdf["activity_type"] == act].sample(1).iloc[0]
        for act in set(locations)
        if (buildings_gdf["activity_type"] == act).any()
    }

    trips = []
    prev_sec = 0
    for i in range(len(locations) - 1):
        dep_sec = parse_time_sec(departure_times[i], prev_sec)
        prev_sec = dep_sec

        o = sampled.get(locations[i])
        d = sampled.get(locations[i + 1])
        if o is None or d is None:
            continue

        trips.append({
            "agent_idx": agent_idx,
            "o_x": o.geometry.x,
            "o_y": o.geometry.y,
            "d_x": d.geometry.x,
            "d_y": d.geometry.y,
            "departure_sec": dep_sec,
        })

    return trips

In [66]:
from tqdm import tqdm

all_trips = []

for result in tqdm(agent_results):
    itinerary = result.get("itinerary")
    agent_idx = result.get("agent_idx")
    if itinerary is None:
        continue
    all_trips.extend(sample_trips(itinerary, joined, agent_idx=agent_idx))

all_trips[:3]

100%|██████████| 96/96 [00:37<00:00,  2.56it/s]


[{'agent_idx': 0,
  'o_x': -87.69009235406418,
  'o_y': 41.801568167410665,
  'd_x': -87.60777492747464,
  'd_y': 41.68528454780155,
  'departure_sec': 28800},
 {'agent_idx': 0,
  'o_x': -87.60777492747464,
  'o_y': 41.68528454780155,
  'd_x': -87.69009235406418,
  'd_y': 41.801568167410665,
  'departure_sec': 54600},
 {'agent_idx': 1,
  'o_x': -87.70391816104369,
  'o_y': 42.01145281456124,
  'd_x': -87.67985368416487,
  'd_y': 41.910667418138054,
  'departure_sec': 28800}]

In [73]:
len(agent_results)

96

In [70]:
import pandas as pd
trips = pd.DataFrame(all_trips)

In [71]:
trips

,agent_idx,o_x,o_y,d_x,d_y,departure_sec
0,0,-87.690092,41.801568,-87.607775,41.685285,28800
1,0,-87.607775,41.685285,-87.690092,41.801568,54600
2,1,-87.703918,42.011453,-87.679854,41.910667,28800
3,1,-87.679854,41.910667,-87.703918,42.011453,34200
4,3,-87.657880,41.715993,-87.607706,41.662973,25200
...,...,...,...,...,...,...
145,96,-87.759078,41.917739,-87.675503,41.933471,64800
146,97,-87.817299,41.996706,-87.729847,41.869264,21600
147,97,-87.729847,41.869264,-87.803219,41.937849,30600
148,98,-87.709067,41.697141,-87.611686,41.687860,25200


In [74]:
trips.to_csv("run/ATLAS_test_trips.csv")